<a href="https://colab.research.google.com/github/KinzaAsif2456/discoverey/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess
import pandas as pd
import numpy as np

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**RULE:**
"Prioritize content where performance has hit a plateau or decline relative to visibility. We flag pages that are either outdated or are ranking well (Top 20) but failing to convert views into clicks. A special high-priority tier exists for 'High Stale Decoy'—very old content that still draws massive impressions, representing our highest-risk/highest-reward refresh opportunities."

**Reason Codes:**

HIGH_STALE_DECOY: Over 6 months old (181+) AND top-tier impressions (>5,000 90d).

CTR_UNDERPERFORM: CTR below 0.5% while ranking in the Top 20.

STALE_CONTENT: Content hasn't been updated in over 90 days.

COMBO_STALE_CTR: Both Staleness and CTR-underperformance signals are present.

**Why I chose these thresholds:**


- I used 90 days because content that has not been updated for more than three months is more likely to become outdated.
- I used a CTR threshold of 0.5% because pages ranking in the top 20 should generally receive more clicks. A lower CTR suggests that the title, meta description, or content may need improvement.
- I also required at least 500 impressions so that very low-traffic pages do not trigger the rule based on noisy data.

In [2]:
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
base_rate = df["is_declining_label"].mean()
print(f"rows: {len(df)}, base decline rate: {base_rate:.3f}")

rows: 30000, base decline rate: 0.542


In [3]:
# Signal 1: staleness (behind FlyRank's refresh flags) ---


                    n  decline_rate
freshness_tier                     
0-30            20480      0.511377
31-90             175      0.588571
91-180           9171      0.611057
181+              174      0.471264


VERDICT: MIXED

Decline increases from 0–30 through 91–180 days, but drops in the 181+ bucket. Because the 181+ bucket contains only 174 pages compared with over 20,000 in newer buckets, this reversal may be due to small sample size rather than a true pattern. Freshness still appears useful, but the oldest bucket should be interpreted cautiously.

In [4]:
# Signal 2: CTR-vs-position (behind the CTR-fix logic) ---
df["low_ctr_flag"] = (
    (df["impressions_90d"] >= 500) &
    (df["avg_position"] > 0) & (df["avg_position"] <= 20) &
    (df["ctr"] < 0.005)
)

ctr_table = (df.groupby("low_ctr_flag")
               .agg(n=("is_declining_label", "size"),
                    decline_rate=("is_declining_label", "mean")))
print(ctr_table)



                  n  decline_rate
low_ctr_flag                     
False         28784      0.532310
True           1216      0.773026


VERDICT: CONFIRMED

Pages with low CTR while ranking in the top 20 have a higher decline rate than pages without this flag. This supports using CTR performance as another signal in the baseline rule.

I selected 5000 impressions as the high visibility threshold because pages with large search exposure but weak engagement represent a bigger opportunity cost. A high-impression stale page can create more missed traffic than a low-volume page.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [9]:

# Thresholds based on Signal Audit
STALE_TIERS = ["91-180", "181+"]
CTR_THRESHOLD = 0.005 # 0.5%
IMPRESSION_DECOY_LEVEL = 5000

# 1. Flag components
df["stale_flag"] = df["freshness_tier"].isin(STALE_TIERS)
df["low_ctr_flag"] = (df["avg_position"] <= 20) & (df["ctr"] < CTR_THRESHOLD) & (df["impressions_90d"] >= 500)
df["is_decoy"] = (df["freshness_tier"] == "181+") & (df["impressions_90d"] >= IMPRESSION_DECOY_LEVEL)

# 2. Score logic (Weights: Decoy = 3, Combo = 2, Single = 1)
def calculate_score(row):
    if row["is_decoy"]: return 3
    if row["stale_flag"] and row["low_ctr_flag"]: return 2
    if row["stale_flag"] or row["low_ctr_flag"]: return 1
    return 0

df["score"] = df.apply(calculate_score, axis=1)

# 3. Reason code implementation (Ensures alignment with Section 1)
def get_reason(row):
    if row["is_decoy"]: return "HIGH_STALE_DECOY"
    if row["stale_flag"] and row["low_ctr_flag"]: return "COMBO_STALE_CTR"
    if row["low_ctr_flag"]: return "CTR_UNDERPERFORM"
    if row["stale_flag"]: return "STALE_CONTENT"
    return "NO_SIGNAL"

df["reason_code"] = df.apply(get_reason, axis=1)

# 4. Action labels
df["action"] = df["score"].apply(lambda s: "REFRESH_NOW" if s >= 2 else ("REVIEW" if s == 1 else "NO_ACTION"))

# Rank by score first, then impressions to prioritize 'big' pages
queue = df.sort_values(["score", "impressions_90d"], ascending=[False, False]).reset_index(drop=True)
queue["rank"] = queue.index + 1

# Export
os.makedirs("work/outputs", exist_ok=True)
out_cols = ["content_id", "client_id", "rank", "score", "reason_code", "action", "is_declining_label"]
queue[out_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"Queue Size: {len(queue)} | Baseline Decline Rate: {df['is_declining_label'].mean():.2%}")

Queue Size: 30000 | Baseline Decline Rate: 54.21%


In [6]:
!head work/outputs/baseline_action_score.csv

content_id,client_id,rank,score,reason_code,action,is_declining_label
content_cf56e2e2e282,client_7f2253d7e2,1,3,HIGH_STALE_DECOY,REFRESH_NOW,1
content_7368877ea310,client_7f2253d7e2,2,3,HIGH_STALE_DECOY,REFRESH_NOW,1
content_1bfaa38ff26c,client_7f2253d7e2,3,3,HIGH_STALE_DECOY,REFRESH_NOW,1
content_0a91db491d14,client_7f2253d7e2,4,3,HIGH_STALE_DECOY,REFRESH_NOW,1
content_5feee3994adb,client_7f2253d7e2,5,3,HIGH_STALE_DECOY,REFRESH_NOW,1
content_c2d929d83eaa,client_7f2253d7e2,6,3,HIGH_STALE_DECOY,REFRESH_NOW,1
content_c8e9d6ab9013,client_19581e27de,7,2,COMBO_STALE_CTR,REFRESH_NOW,1
content_825a9788af8d,client_4e07408562,8,2,COMBO_STALE_CTR,REFRESH_NOW,1
content_8ba781dafa55,client_8527a891e2,9,2,COMBO_STALE_CTR,REFRESH_NOW,1


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [7]:
# Code to display top 20 for review
review_cols = ["content_id", "rank", "score", "reason_code", "action",
               "freshness_tier", "avg_position", "ctr", "impressions_90d"]
print(queue[review_cols].head(20).to_string(index=False))

          content_id  rank  score      reason_code      action freshness_tier  avg_position  ctr  impressions_90d
content_cf56e2e2e282     1      3 HIGH_STALE_DECOY REFRESH_NOW           181+          19.7 0.15            61678
content_7368877ea310     2      3 HIGH_STALE_DECOY REFRESH_NOW           181+          24.8 0.13            59472
content_1bfaa38ff26c     3      3 HIGH_STALE_DECOY REFRESH_NOW           181+          22.2 0.23            25715
content_0a91db491d14     4      3 HIGH_STALE_DECOY REFRESH_NOW           181+          10.5 0.49            13299
content_5feee3994adb     5      3 HIGH_STALE_DECOY REFRESH_NOW           181+          39.0 0.01             7812
content_c2d929d83eaa     6      3 HIGH_STALE_DECOY REFRESH_NOW           181+          17.9 0.20             7558
content_c8e9d6ab9013     7      2  COMBO_STALE_CTR REFRESH_NOW         91-180           9.7 0.00           208678
content_825a9788af8d     8      2  COMBO_STALE_CTR REFRESH_NOW         91-180           

**Top-20 Review (Data-Bound SME Critique)**

| Rank | Content ID | Reason Code | Key Metrics | Why it's there? | What would make it wrong? |
|------|------------|-------------|-------------|-----------------|---------------------------|
| 1 | cf56e2e2e282 | HIGH_STALE_DECOY | 61.6k Imps, 15% CTR | Massive exposure but hasn't been touched in 6+ months. | Answer might be in the Google snippet (Featured Snippet), so users don't need to click. |
| 2 | 7368877ea310 | HIGH_STALE_DECOY | 59.4k Imps, 24.8 Pos | Old content on Page 2 with massive impression volume. | Page-2 results rarely get clicked; low CTR is likely position-based, not quality-based. |
| 3 | 1bfaa38ff26c | HIGH_STALE_DECOY | 25.7k Imps, 22.2 Pos | Very high impression decoy with stale status. | High volume could be "noise" from accidental ranking for a highly searched irrelevant keyword. |
| 4 | 0a91db491d14 | HIGH_STALE_DECOY | 13.2k Imps, 10.5 Pos | Hovering on Page 1/2 boundary with stale data. | This is likely a reference/directory page; people look at the title but don't intend to enter. |
| 5 | 5feee3994adb | HIGH_STALE_DECOY | 7.8k Imps, 39.0 Pos | Stale content on Page 4 still pulling visibility. | Being on Page 4 explains the 0.01 CTR; content isn't necessarily bad, it's just physically hidden. |
| 6 | c2d929d83eaa | HIGH_STALE_DECOY | 7.5k Imps, 17.9 Pos | Decaying visibility for a page with solid exposure. | SERP image blocks or maps may be "pushing" this result down the mobile screen. |
| 7 | c8e9d6ab9013 | COMBO_STALE_CTR | 208.6k Imps, 0.00 CTR | **Flagged:** 200k+ views but literal zero clicks at rank 9. | **Tracking Error:** It is statistically unlikely to have 0 clicks with 200k impressions. The pixel is likely broken. |
| 8 | 825a9788af8d | COMBO_STALE_CTR | 16.7k Imps, 5.6 Pos | Rank 5.6 with zero click conversion. | Competitive brand dominance: If Google lists Amazon above us, everyone clicks Amazon regardless. |
| 9 | 8ba781dafa55 | COMBO_STALE_CTR | 16.1k Imps, 9.0 Pos | High visibility on page 1, failing to engage. | Topic is broad (top of funnel). Users browse the SERP but aren't ready to click a specific guide yet. |
| 10 | d3aaf7d5f2fc | COMBO_STALE_CTR | 7.7k Imps, 8.3 Pos | Good visibility at Rank 8 but stagnant. | Page might answer an "instant info" query (e.g., date of an event) so clicks are unnecessary. |
| 11 | 1de025a8c508 | COMBO_STALE_CTR | 7.0k Imps, 16.0 Pos | Stale page buried on the bottom of page 1. | Too much "Reviewer Bloat": Users skip long lists and stick to the top 3 organic results. |
| 12 | e15ede72712d | COMBO_STALE_CTR | 6.7k Imps, 16.1 Pos | Similar to above; stale signal + high decoy potential. | Metadata Title/Description might be truncated or showing a dated "2023" year tag in 2024/2025. |
| 13 | 5195668f06db | COMBO_STALE_CTR | 6.6k Imps, 5.2 Pos | Very high position (Rank 5) but 0 clicks. | **Intent Mismatch:** We rank for a word that sounds like what the user wants but isn't it. |
| 14 | c65ee459f729 | COMBO_STALE_CTR | 6.5k Imps, 17.4 Pos | Underperforming visibility despite good rank. | CTR measurement might be lagging; recent ranking jump may not show clicks yet. |
| 15 | 9648b7053d6f | COMBO_STALE_CTR | 6.5k Imps, 9.3 Pos | At the edge of the Top 10 with staleness risk. | Users are likely bouncing to Ads which are crowding the Rank 1–10 area. |
| 16 | 9ee24f9f28c3 | COMBO_STALE_CTR | 5.7k Imps, 8.3 Pos | Clear underperformer at Rank 8. | **Technical Issue:** Page may be very slow to load (Core Web Vitals), preventing clicks from logging. |
| 17 | 28a161c4e8c5 | COMBO_STALE_CTR | 5.2k Imps, 18.5 Pos | Visibility is high but age is a massive concern. | This might be a legacy news post that was popular once; refreshing won't restore old news intent. |
| 18 | b331a2c7719d | COMBO_STALE_CTR | 5.1k Imps, 11.5 Pos | Just missed the Top 10. Low conversion. | If this is an evergreen topic, it's just slightly weak—content update might not move the needle. |
| 19 | 7722f40b12c1 | COMBO_STALE_CTR | 5.0k Imps, 8.3 Pos | High Page 1 visibility not capturing market share. | Content depth might be the issue; users want a deep-dive and the title looks like a thin guide. |
| 20 | 92ca79f8b232 | COMBO_STALE_CTR | 4.6k Imps, 7.5 Pos | Score of 2 indicates clear intervention needed. | High-volatility query; competitors might be updating their prices/dates daily while we are stale. |

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [8]:
print("Columns used in scoring: freshness_tier, avg_position, impressions_90d, ctr")
print("trend_direction / trend_pct used only to build is_declining_label (the label) —",
      "never inside stale_flag, low_ctr_flag, score, reason_code, or action.")

borderline = queue[queue["score"] == 1].sort_values("impressions_90d").head(3)
print(borderline[review_cols])

Columns used in scoring: freshness_tier, avg_position, impressions_90d, ctr
trend_direction / trend_pct used only to build is_declining_label (the label) — never inside stale_flag, low_ctr_flag, score, reason_code, or action.
                 content_id   rank  score    reason_code  action  \
10084  content_901b40631379  10085      1  STALE_CONTENT  REVIEW   
10019  content_00b467976cbd  10020      1  STALE_CONTENT  REVIEW   
9976   content_1e51704e6f13   9977      1  STALE_CONTENT  REVIEW   

      freshness_tier  avg_position  ctr  impressions_90d  
10084         91-180          15.0  0.0                1  
10019         91-180          10.0  0.0                1  
9976          91-180           9.0  0.0                1  


**WEAK PICKS:**

**Case 1:**
Some pages are very stale but receive very few impressions. Refreshing them may not produce enough benefit to justify the work.

**Case 2:**
Some pages have CTR only slightly below the 0.5% threshold. Their low CTR could simply be normal week-to-week variation rather than a true problem.

**Case 3:**
Some pages rank well but belong to evergreen topics. Even though they are old, updating them may not improve performance and could introduce unnecessary risk.


**LEAKAGE CHECK:**
I confirmed that is_declining_label (and the trend_direction it was derived from) is not used to calculate the score. We only used freshness_tier, avg_position, and ctr for the rule logic. This ensures our baseline isn't 'cheating' by looking at the future trend.

## Conclusion

This baseline uses two simple and explainable signals: content freshness and CTR performance. Both signals were checked before building the rule and showed evidence that they are useful indicators. The scoring system is easy to interpret and avoids using future information or label-derived features. Although this rule will not identify every declining page, it provides a strong baseline that can be compared with a machine learning model in the next stage of the project.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.